# Paper figure & stats — *PWS reads zero in snow / ice*

Evidence for the claim that consumer Weather Underground tipping-bucket gauges fail to
capture frozen precipitation, while NWS ASOS heated gauges and Mesonet weighing gauges
do. All cells are knobs + a single `WN.*` call; the analysis logic lives in
`src/analysis/nycmesh_utils.py`.

Sections:

1. **Stats — headline undercatch summary** (single row tells the whole story)
2. **Stats — full stratified table** (mean / median / pct-zero / total per ASOS precip_category)
3. **Figure — hourly distribution** by precip_category (box plots, paper figure 1)
4. **Per-PWS-station response** (under-catch is system-wide, not 1–2 bad sensors)
5. **Figure — event zoom A** (Jan 2026 snowstorm, paper figure 2)
6. **Figure — event zoom B** (Feb 2024 storm, supplemental)
7. **Figure — event zoom C** (Feb 2026 follow-on, supplemental)


## §1. Setup


In [ ]:
import sys, warnings
from pathlib import Path
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# Avoid HDF cache evictions across the big merged file.
xr.set_options(file_cache_maxsize=512)

REPO_ROOT = Path('../../../').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))
from analysis import nycmesh_utils as WN

# ── Paths ──────────────────────────────────────────────────────────────
OUT = REPO_ROOT / 'dataset' / 'raw' / 'full' / 'outputs'
ASOS_NC = OUT / 'asos_2023-10-01_2026-04-23.nc'
PWS_NC  = OUT / 'pws_wu_merged_2023-10-29_2026-04-24_qc.nc'
MESO_NC = OUT / 'mesonet_2023-08-01_2026-03-04.nc'

# Common window — use the full overlap so we maximise the snow-hour count.
WINDOW = (pd.Timestamp('2023-10-29'), pd.Timestamp('2026-04-24'))

# Flip to a Path() to save figures for the paper; default keeps work in-memory.
FIG_DIR = None    # e.g. REPO_ROOT / 'dataset' / 'examples' / 'paper_figs'

print(f'Window: {WINDOW[0].date()} → {WINDOW[1].date()}  '
      f'({(WINDOW[1]-WINDOW[0]).days} days)')


## §2. Load networks


In [ ]:
NETWORKS = WN.load_weather_networks(asos_nc=ASOS_NC, pws_nc=PWS_NC, meso_nc=MESO_NC)


## §3. Headline statistic — the paper one-liner

For each ASOS-classified precipitation category, what fraction of liquid mass does
each network capture? The `snow` row is the central paper number.


In [ ]:
undercatch = WN.pws_undercatch_summary(NETWORKS, window=WINDOW, detect_threshold_mm=0.5)


## §4. Full stratified table

Per-category mean / median / pct-zero / total per network. Drop into the paper
appendix or methods section.


In [ ]:
stats = WN.rainfall_by_precip_category(NETWORKS, window=WINDOW)


## §5. Figure — hourly distribution by category

Box plots (log-y, wet hours only) showing the distributional collapse of PWS
hourly rainfall during ASOS-classified snow hours.


In [ ]:
save = (FIG_DIR / 'fig1_pws_distribution_by_category.pdf') if FIG_DIR else None
WN.plot_hourly_distribution_by_category(
    NETWORKS,
    categories=('rain', 'snow', 'mix'),
    window=WINDOW,
    save_path=save,
)
plt.show()


## §6. Per-PWS-station response

Sanity check: under-catch is uniform across the 80 PWS stations, not driven by a
handful of bad sensors. Median per-station mean rainfall in ASOS snow hours is
orders of magnitude below rain hours.


In [ ]:
per_station = WN.pws_station_snow_response(NETWORKS, window=WINDOW, min_n_snow=20)


## §7. Event zoom A — Jan–Feb 2026 snowstorms *(paper figure 2)*

This 2-week stretch drove the entire 2026 ASOS-vs-PWS divergence: ASOS
reports 50–100+ mm/day of liquid-equivalent while the PWS network mean is
effectively 0 mm/day, with Mesonet `snow_depth` holding at 15–25 cm.


In [ ]:
save = (FIG_DIR / 'fig2_jan2026_snowstorm.pdf') if FIG_DIR else None
WN.plot_pws_snow_event_compare(
    NETWORKS, '2026-01-23', '2026-02-06',
    title='Jan–Feb 2026 snowstorms — ASOS records 100+ mm/day, PWS records 0',
    save_path=save,
)
plt.show()


## §8. Event zoom B — Jan 2025 snow event *(different winter)*


In [ ]:
save = (FIG_DIR / 'fig_sup_jan2025_event.pdf') if FIG_DIR else None
WN.plot_pws_snow_event_compare(
    NETWORKS, '2025-01-21', '2025-01-28',
    title='Jan 2025 snow event — gap visible in a different winter',
    save_path=save,
)
plt.show()


## §9. Event zoom C — Jan 2024 wet-snow contrast

Some snow events show only a partial PWS gap — usually wet snow events near
freezing (>~0°C), where bucket warmth melts incoming flakes. Include this in
the paper to acknowledge the gap is **temperature-dependent**, not absolute.


In [ ]:
save = (FIG_DIR / 'fig_sup_jan2024_contrast.pdf') if FIG_DIR else None
WN.plot_pws_snow_event_compare(
    NETWORKS, '2024-01-13', '2024-01-21',
    title='Jan 2024 — wet-snow contrast (PWS partially catches, gap is smaller)',
    save_path=save,
)
plt.show()
